# Paper 3 full temporal grouping sensitivity

This notebook runs the approved post-hoc 12-/24-hour regrouping comparison on your original source rows, using saved detectors. It also retains the historical six-hour baseline and performs strict dataset-role-excluded predictive validation. **It does not retrain detectors or resample raw datasets.**

**Run:** upload this notebook to Colab, choose Runtime → Run all, and allow Drive access. Use the same CPU runtime/software as the successful pilot where possible. The pilot projects about 5.9 hours of scoring; loading and 60 model replay checks add time, and other datasets may differ. Keep only one running copy.

**Resume:** if disconnected, reopen this same notebook and choose Runtime → Run all. The default run folder is fixed, so verified completed groups are reused automatically. A partially calculated group is recomputed. Software/code/plan changes are rejected for the existing folder. Do not delete completed checkpoints.

**Completion:** wait for FULL_PASS and an automatic ZIP download. Upload the ZIP here. COMPLETED_WITH_RANK_DEFICIENCY requires inspection and is not a full-rank pass. If a check fails, the notebook stops and saves an error report. The final export cell can also be run separately to download partial progress.

Outputs: MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet/supplementary_temporal_grouping_20261023/run_v1. The original experiment folders remain unchanged. This new grouping study remains post hoc; narrower raw-flow windows, independent datasets and operational prevalence are outside its scope.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import importlib.util
import subprocess
import sys

drive.mount('/content/drive')
missing = [p for p in ['pyarrow','xgboost'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*missing],check=True)
PAPER3_ROOT = Path('/content/drive/MyDrive/research/researchdata/NIDS_Research/Paper3_DataSet')
if not PAPER3_ROOT.is_dir():
    raise FileNotFoundError(PAPER3_ROOT)
# Leave this unchanged to resume the same run after a disconnection.
RUN_NAME = 'run_v1'
OUTPUT = PAPER3_ROOT / 'supplementary_temporal_grouping_20261023' / RUN_NAME
OUTPUT.mkdir(parents=True,exist_ok=True)
print('Run and resume folder:',OUTPUT)


In [ ]:
FULL_PLAN = "# Full temporal grouping sensitivity plan\n\nThis is post hoc after original target outcomes, prior supplementary comparisons and a one-source timing pilot were known. Record this plan before new full-run importance calculations. Preserve all original artifacts.\n\nScope: four original NetFlow-v3 source datasets, seeds 2027–2031, three original fitted classifiers, 49 predictors, unchanged source preprocessors and source thresholds. No detector retraining. Reuse the same saved environmental rows: merge original six-hour UTC block identifiers into aligned 12- and 24-hour groups, retaining every row. No exact timestamps are recovered, no shorter intervals are evaluated and no new raw-flow panels are sampled. Different group counts, group sizes and precision are part of this bounded regrouping sensitivity; it does not establish optimal temporal resolution.\n\nBefore each model is used, verify historical hashes and schema, replay its full source-test probabilities within maximum absolute 1e-7 and identical binary predictions, and replay the first original six-hour environmental importance group with original random states. Require three-repeat importance agreement within 1e-8 and baseline balanced-accuracy agreement within 1e-12. Record actual software and load warnings. Resume only with identical software/code/plan/input identity. Empirical replay is numerical compatibility evidence, not proof of unrestricted cross-version portability.\n\nFor each width, construct pseudo-groups by deterministic within-class shuffling of the original environmental rows, preserving each actual group's exact size and benign/attack counts. All original groups contain at least 2,000 observations per class. Seed pseudo-shuffling from source dataset, seed and a canonical actual-membership fingerprint, not nominal width. Thus identical actual partitions receive identical pseudo-partitions. For each model and group, calculate all 49 permutation importances with three repeats and n_jobs=1 outside the original estimator. Seed the permutations from dataset, seed, model, partition type and the ordered selected-row fingerprint, not width. Cache identical row sets under the same model/partition. Do not import pilot scores because the full procedure uses these canonical random states.\n\nAt each width and source-seed-model unit, mean pairwise Spearman rank correlation over all group pairs defines environmental and sampling stability; ties receive average ranks and constant rank vectors contribute zero. Do not weight pairs by group size. Environmental instability is 1 minus environmental stability; sampling instability is 1 minus sampling stability; the gap is sampling stability minus environmental stability. Record undefined correlations, group counts, pair counts and class sizes. Keep negative importance/gap values. With only two groups, one pair is still reported and explicitly has limited sensitivity. Retain all source/model results regardless of direction.\n\nUse the historical six-hour scores unchanged as baseline. Their original random assignments and importance random states differ from the new canonical procedure; contrasts against six hours therefore also contain Monte Carlo differences. The 12-/24-hour CSE-CIC-IDS2018 memberships are identical in the audit and must give exactly equal scores. These are not independent sensitivity replications. Save/hash all source warnings before reading the saved transfer-loss table for the following analysis. This ordering is a record of this post-hoc calculation, not a historical pre-label freeze.\n\nPredictive analysis: at each width (6, 12, 24 hours), retain six fixed ordinary least-squares specifications: mean only, model identity only, identity plus sampling instability, identity plus environmental instability, identity plus gap, identity plus both primary warnings. Exclude the held-out dataset from both source and target roles in meta-regression training. Training losses average two targets and testing losses three. All-seed folds contain 45 training and 15 testing units. Include all five seed-deletion runs, with 36/12 units per fold. Fit continuous scaling only on training units. No tuning or prediction clipping. Report all pooled and per-dataset MAE/RMSE results; positive improvement is model-only MAE minus candidate MAE within the same width/run/fold. Verify the historical six-hour model-only and primary-warning MAEs against prior saved analyses. Independently check OLS predictions with NumPy. Report rank deficiency if any; do not silently remove predictors or claim a full-rank pass.\n\nReport descriptive Spearman correlations on the 12 source-model units at each width, with no new hypothesis tests, p-values, confidence intervals, power or equivalence claims. No predictive superiority is established from these overlapping folds. This work does not address independent new datasets, natural attack prevalence, attack-family transfer or genuine refitted cross-validation variance.\n\nCheckpoints: write each group atomically with a payload checksum and code/software/input identity. Verify completed checkpoints before reuse. Keep partial progress on Drive, with a single runtime writing a run folder. Store all calculations separately from frozen folders. A full pass requires all group references and 180 source-seed-model-width scores, 432 meta-regression fits and 5,400 OOF predictions. The archived raw pipeline is not reproduced by this run.\n"
FULL_SCRIPT = '"""Post-hoc temporal regrouping on original sampled source rows. No retraining."""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,itertools,json,platform,time,warnings\nimport numpy as np,pandas as pd,scipy,sklearn,joblib\nfrom sklearn.inspection import permutation_importance\nfrom sklearn.metrics import balanced_accuracy_score\nfrom sklearn.linear_model import LinearRegression\n\nDATASETS=[\'NF_BoT_IoT_v3\',\'NF_CSE_CIC_IDS2018_v3\',\'NF_ToN_IoT_v3\',\'NF_UNSW_NB15_v3\']\nMODELS=[\'LogisticRegression\',\'RandomForest\',\'XGBoost\']\nKEY=[\'source_dataset\',\'seed\',\'model\']\nWARN=[\'sampling_instability\',\'environmental_instability\',\'sampling_environment_gap\']\nSPECS={\'mean_only\':[],\'model_only\':[],\'model_plus_sampling\':WARN[:1],\n       \'model_plus_environment\':WARN[1:2],\'model_plus_gap\':WARN[2:],\n       \'model_plus_both_primary\':WARN[1:]}\n\ndef canonical(x):return json.dumps(x,sort_keys=True,separators=(\',\',\':\'),allow_nan=False).encode()\ndef digest(x):return hashlib.sha256(canonical(x)).hexdigest()\ndef sha(path):\n    h=hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\ndef state(*parts):return int.from_bytes(hashlib.sha256(\'|\'.join(map(str,parts)).encode()).digest()[:4],\'little\')\ndef write(path,obj):\n    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True);tmp=p.with_suffix(p.suffix+\'.tmp\')\n    tmp.write_text(json.dumps(obj,indent=2,allow_nan=False)+\'\\n\');tmp.replace(p)\ndef seal(path,obj):\n    obj=dict(obj);obj.pop(\'payload_sha256\',None);obj[\'payload_sha256\']=digest(obj);write(path,obj)\ndef readsealed(path):\n    x=json.loads(Path(path).read_text());h=x.pop(\'payload_sha256\');assert digest(x)==h,\'Checkpoint checksum mismatch\';return x\ndef score_at(threshold):\n    def score(model,x,y):return balanced_accuracy_score(y,(model.predict_proba(x)[:,1]>=threshold).astype(np.int8))\n    return score\ndef paired_stability(vectors):\n    ranks=[pd.Series(-np.asarray(v)).rank(method=\'average\').to_numpy() for v in vectors]\n    values=[];undefined=0\n    for a,b in itertools.combinations(ranks,2):\n        if np.std(a)==0 or np.std(b)==0:values.append(0.);undefined+=1\n        else:values.append(float(np.corrcoef(a,b)[0,1]))\n    assert len(values)>0\n    return float(np.mean(values)),undefined,len(values)\ndef matched_groups(frame,hours,dataset,seed):\n    actual=frame.block_id_ms.to_numpy(np.int64)//(hours*3600000)*(hours*3600000)\n    starts=np.sort(np.unique(actual));codes=np.searchsorted(starts,actual)\n    # Same membership -> same pseudo-assignment and permutation states at both widths.\n    membership=digest({\'row_ids\':frame.row_id.astype(\'int64\').tolist(),\'actual_codes\':codes.tolist()})\n    pseudo=np.full(len(frame),-1,dtype=np.int64);y=frame.Label.to_numpy(np.int8)\n    rng=np.random.default_rng(state(\'TEMPORAL_REGROUPING_FULL_V1\',dataset,seed,membership,\'pseudo\'))\n    for label in [0,1]:\n        positions=rng.permutation(np.flatnonzero(y==label));offset=0\n        for code in range(len(starts)):\n            count=int(np.sum((codes==code)&(y==label)));pseudo[positions[offset:offset+count]]=code;offset+=count\n        assert offset==len(positions)\n    assert np.all(pseudo>=0)\n    for code in range(len(starts)):\n        a=np.bincount(y[codes==code],minlength=2);b=np.bincount(y[pseudo==code],minlength=2)\n        assert np.array_equal(a,b) and np.all(a>=2000)\n    return codes,pseudo,membership,starts\n\ndef predictive_analysis(root,out,units):\n    # All new source scores are saved and hashed before reading the saved transfer table.\n    units.to_csv(out/\'source_warning_scores_all_widths.csv\',index=False)\n    frozen=sha(out/\'source_warning_scores_all_widths.csv\')\n    write(out/\'source_scores_record.json\',{\'created_utc\':datetime.now(timezone.utc).isoformat(),\n        \'sha256\':frozen,\'post_hoc_after_original_outcomes_known\':True,\'not_a_historical_pre_label_seal\':True})\n    transfer_path=root/\'Paper3_Final_Evaluation/transfer_analysis_all_180.csv\'\n    transfers=pd.read_csv(transfer_path);assert len(transfers)==180\n    predictions=[];diagnostics=[];associations=[]\n    for hours in [6,12,24]:\n        controls=units[units.hours==hours].rename(columns={\'dataset\':\'source_dataset\'})\n        assert len(controls)==60 and not controls.duplicated(KEY).any()\n        f=transfers.drop(columns=WARN,errors=\'ignore\').merge(controls[KEY+WARN],on=KEY,validate=\'many_to_one\')\n        assert len(f)==180 and np.isfinite(f[WARN+[\'balanced_accuracy_loss\']]).all().all()\n        def aggregate(a):\n            return a.groupby(KEY,as_index=False).agg(loss=(\'balanced_accuracy_loss\',\'mean\'),**{c:(c,\'first\') for c in WARN})\n        combined=aggregate(f).groupby([\'source_dataset\',\'model\'],as_index=False)[[\'loss\']+WARN].mean()\n        assert len(combined)==12\n        for c in WARN:\n            rho=combined[c].rank().corr(combined.loss.rank())\n            associations.append({\'hours\':hours,\'warning\':c,\'units\':12,\'spearman_rho\':None if pd.isna(rho) else float(rho),\'inference\':\'descriptive_only_no_p_value\'})\n        for omit in [None]+list(range(2027,2032)):\n            sub=f if omit is None else f[f.seed!=omit];run=\'all_seeds\' if omit is None else \'omit_seed:\'+str(omit)\n            for held in sorted(sub.source_dataset.unique()):\n                train=sub[(sub.source_dataset!=held)&(sub.target_dataset!=held)];test=sub[sub.source_dataset==held]\n                assert held not in set(train.source_dataset)|set(train.target_dataset)\n                assert train.groupby(KEY).target_dataset.nunique().eq(2).all() and test.groupby(KEY).target_dataset.nunique().eq(3).all()\n                t,e=aggregate(train),aggregate(test)\n                assert len(t)==(45 if omit is None else 36) and len(e)==(15 if omit is None else 12)\n                for spec,cols in SPECS.items():\n                    a=[np.ones(len(t))];b=[np.ones(len(e))]\n                    if spec!=\'mean_only\':\n                        for model in [\'RandomForest\',\'XGBoost\']:\n                            a.append(t.model.eq(model).to_numpy(float));b.append(e.model.eq(model).to_numpy(float))\n                    for c in cols:\n                        mu=float(t[c].mean());sd=float(t[c].std(ddof=0));sd=sd if sd>0 else 1.\n                        a.append((t[c].to_numpy()-mu)/sd);b.append((e[c].to_numpy()-mu)/sd)\n                    a,b=np.column_stack(a),np.column_stack(b)\n                    pred=LinearRegression(fit_intercept=False).fit(a,t.loss).predict(b)\n                    direct=b@np.linalg.lstsq(a,t.loss,rcond=None)[0];diff=float(np.max(abs(pred-direct)));assert diff<1e-9\n                    rank=int(np.linalg.matrix_rank(a));condition=float(np.linalg.cond(a))\n                    diagnostics.append({\'hours\':hours,\'run\':run,\'held_out\':held,\'specification\':spec,\'rank\':rank,\'columns\':a.shape[1],\n                        \'full_rank\':rank==a.shape[1],\'condition_number\':condition if np.isfinite(condition) else None,\'independent_difference\':diff})\n                    for (_,r),v in zip(e.iterrows(),pred):\n                        predictions.append({\'hours\':hours,\'run\':run,\'held_out\':held,\'seed\':int(r.seed),\'model\':r.model,\n                            \'specification\':spec,\'actual_loss\':float(r.loss),\'predicted_loss\':float(v)})\n    pred=pd.DataFrame(predictions);rows=[]\n    for level,keys in [(\'overall\',[\'hours\',\'run\',\'specification\']),(\'fold\',[\'hours\',\'run\',\'held_out\',\'specification\'])]:\n        for key,g in pred.groupby(keys):\n            error=g.actual_loss-g.predicted_loss\n            rows.append({**dict(zip(keys,key)),\'level\':level,\'units\':len(g),\'mae\':float(abs(error).mean()),\'rmse\':float(np.sqrt((error**2).mean()))})\n    scores=pd.DataFrame(rows).fillna(\'ALL\');scope=[\'hours\',\'run\',\'level\',\'held_out\']\n    reference=scores[scores.specification==\'model_only\'].set_index(scope).mae\n    scores[\'mae_improvement_vs_model_only\']=[float(reference.loc[tuple(r[k] for k in scope)])-r.mae for _,r in scores.iterrows()]\n    old=scores[(scores.hours==6)&(scores.run==\'all_seeds\')&(scores.level==\'overall\')].set_index(\'specification\')\n    assert abs(old.loc[\'model_only\',\'mae\']-0.05009957252970506)<1e-10\n    assert abs(old.loc[\'model_plus_both_primary\',\'mae\']-0.06689810412254844)<1e-10\n    assert len(pred)==5400 and len(diagnostics)==432\n    pred.to_csv(out/\'strict_validation_predictions.csv\',index=False);scores.to_csv(out/\'strict_validation_scores.csv\',index=False)\n    write(out/\'strict_validation_diagnostics.json\',diagnostics);write(out/\'descriptive_associations.json\',associations)\n    return {\'predictions\':len(pred),\'regression_fits\':len(diagnostics),\'rank_deficient_fits\':sum(not x[\'full_rank\'] for x in diagnostics),\n            \'source_scores_sha256\':frozen,\'saved_transfer_table_sha256\':sha(transfer_path),\'six_hour_reference_reproduced\':True}\n\ndef run_full(root,out,plan_text,script_text):\n    import xgboost\n    root,out=Path(root),Path(out);out.mkdir(parents=True,exist_ok=True)\n    for name,content in [(\'PLAN.md\',plan_text),(\'temporal_grouping_full.py\',script_text)]:\n        path=out/name\n        if path.exists() and path.read_text()!=content:raise RuntimeError(\'Existing run uses different plan/code. Use its original notebook or a new folder.\')\n        if not path.exists():path.write_text(content)\n    software={\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__,\'scipy\':scipy.__version__,\n              \'sklearn\':sklearn.__version__,\'joblib\':joblib.__version__,\'xgboost\':xgboost.__version__}\n    session={\'status\':\'RUNNING\',\'started_utc\':datetime.now(timezone.utc).isoformat(),\'software\':software,\n             \'plan_sha256\':sha(out/\'PLAN.md\'),\'script_sha256\':sha(out/\'temporal_grouping_full.py\'),\n             \'scope\':\'post_hoc_regrouping_of_original_sampled_rows\',\'detector_retraining\':False,\'input_hashes\':{},\'warnings\':[]}\n    identity={\'software\':software,\'plan_sha256\':session[\'plan_sha256\'],\'script_sha256\':session[\'script_sha256\']}\n    lock=out/\'run_identity.json\'\n    if lock.exists():assert readsealed(lock)==identity,\'Software/code/plan differs from existing run. Restore the recorded runtime or use a new run folder.\'\n    else:seal(lock,identity)\n    def record(path,expected=None):\n        value=sha(path)\n        if expected is not None:assert value==expected,\'Historical input hash mismatch: \'+str(path)\n        session[\'input_hashes\'][str(path.relative_to(root))]=value;return value\n    def load(path,expected):\n        record(path,expected)\n        with warnings.catch_warnings(record=True) as caught:\n            warnings.simplefilter(\'always\');result=joblib.load(path)\n        session[\'warnings\'].extend({\'file\':path.name,\'warning\':str(x.message)} for x in caught)\n        return result\n    rows=[];group_rows=[];completed_refs=0;new_jobs=0;reused_jobs=0;total_refs=0\n    # Source-only preflight records the whole intended workload before any new PFI.\n    for ds,seed in itertools.product(DATASETS,range(2027,2032)):\n        f=root/\'Paper3_Sanitized\'/ds/f\'seed_{seed}\'/\'environment.parquet\'\n        q=pd.read_parquet(f,columns=[\'block_id_ms\'])\n        for hours in [12,24]:total_refs+=len(np.unique(q.block_id_ms.to_numpy(np.int64)//(hours*3600000)))*2*3\n    session[\'total_group_references\']=total_refs;write(out/\'progress.json\',session)\n    for dataset,seed in itertools.product(DATASETS,range(2027,2032)):\n        base=root/\'Paper3_Source_Models\'/dataset/f\'seed_{seed}\';meta_path=base/\'source_seed_complete.json\'\n        record(meta_path);meta=json.loads(meta_path.read_text())\n        schema=root/\'Paper3_Materialized/feature_schema_v1.json\';record(schema,meta[\'feature_schema_sha256\'])\n        features=json.loads(schema.read_text())[\'predictors_in_order\'];assert len(features)==49\n        env_path=root/\'Paper3_Sanitized\'/dataset/f\'seed_{seed}\'/\'environment.parquet\'\n        sanitization=env_path.with_name(\'sanitization_manifest.json\');record(sanitization,meta[\'input_sanitization_manifest_sha256\'])\n        audit=json.loads(sanitization.read_text());env_hash=record(env_path,audit[\'output_files\'][\'environment\'][\'sha256\'])\n        test_path=env_path.with_name(\'source_test.parquet\');record(test_path,audit[\'output_files\'][\'source_test\'][\'sha256\'])\n        env=pd.read_parquet(env_path);test=pd.read_parquet(test_path)\n        assert env.row_id.is_unique and test.row_id.is_unique and not set(env.row_id)&set(test.row_id)\n        assert np.all(env.block_id_ms.to_numpy(np.int64)%(6*3600000)==0)\n        assert env.Label.isin([0,1]).all() and test.Label.isin([0,1]).all()\n        prep=load(base/\'source_preprocessor.joblib\',meta[\'preprocessor\'][\'sha256\']);assert list(prep[\'features\'])==features\n        def transform(frame):\n            raw=frame[features].to_numpy(np.float32);x=prep[\'imputer\'].transform(raw).astype(np.float32)\n            xs=prep[\'logistic_regression_scaler\'].transform(x).astype(np.float32)\n            assert np.isfinite(x).all() and np.isfinite(xs).all();return x,xs\n        ex,tx=transform(env),transform(test);y=env.Label.to_numpy(np.int8)\n        groups={hours:matched_groups(env,hours,dataset,seed) for hours in [12,24]}\n        for model_name in MODELS:\n            info=meta[\'models\'][model_name];artifact=load(base/(model_name+\'_model.joblib\'),info[\'files\'][\'model\'][\'sha256\'])\n            model=artifact[\'model\'];threshold=float(artifact[\'threshold\']);scaled=int(bool(artifact[\'uses_standard_scaler\']))\n            assert abs(threshold-info[\'source_test_metrics\'][\'threshold\'])<1e-15\n            x=ex[scaled];xt=tx[scaled]\n            oldpath=root/\'Paper3_Source_Stability\'/dataset/f\'seed_{seed}\'/model_name/\'stability_complete.json\'\n            record(oldpath);original=json.loads(oldpath.read_text())\n            pfipath=oldpath.with_name(\'permutation_importance.parquet\');record(pfipath,original[\'permutation_importance_sha256\'])\n            storedpath=base/(model_name+\'_source_test_predictions.parquet\');record(storedpath,info[\'files\'][\'source_test_predictions\'][\'sha256\'])\n            check_identity={**identity,\'model_sha256\':sha(base/(model_name+\'_model.joblib\')),\'preprocessor_sha256\':sha(base/\'source_preprocessor.joblib\'),\n                \'environment_sha256\':env_hash,\'source_test_sha256\':sha(test_path),\'saved_predictions_sha256\':sha(storedpath),\'saved_importance_sha256\':sha(pfipath)}\n            check_path=out/\'compatibility\'/dataset/f\'seed_{seed}\'/(model_name+\'.json\')\n            if check_path.exists():\n                check=readsealed(check_path);assert check[\'identity\']==check_identity and check[\'status\']==\'PASS\'\n            else:\n                print(\'Checking original model/PFI:\',dataset,seed,model_name,flush=True)\n                saved=pd.read_parquet(storedpath).set_index(\'row_id\');assert saved.index.is_unique and len(saved)==len(test)\n                saved=saved.loc[test.row_id];assert np.array_equal(saved.y_true.to_numpy(),test.Label.to_numpy())\n                probability=model.predict_proba(xt)[:,1];pdiff=float(np.max(abs(probability-saved.probability_attack.to_numpy())))\n                assert pdiff<=1e-7 and np.array_equal((probability>=threshold).astype(np.int8),saved.y_pred.to_numpy()),\'Source-test replay mismatch\'\n                group=str(sorted(env.actual_environment_id.unique())[0]);positions=np.flatnonzero(env.actual_environment_id.astype(str).to_numpy()==group)\n                old=pd.read_parquet(pfipath);ref=old[(old.group_type==\'environmental\')&(old.group_id.astype(str)==group)].set_index(\'feature\').loc[features]\n                assert len(ref)==49 and len(positions)==4000\n                pfi=permutation_importance(model,x[positions],y[positions],scoring=score_at(threshold),n_repeats=3,\n                    random_state=state(dataset,seed,model_name,\'environmental\',group,\'PFI\'),n_jobs=1)\n                difference=float(np.max(abs(pfi.importances-ref[[\'repeat_1\',\'repeat_2\',\'repeat_3\']].to_numpy())))\n                bd=abs(score_at(threshold)(model,x[positions],y[positions])-float(ref.baseline_balanced_accuracy.iloc[0]))\n                assert difference<=1e-8 and bd<=1e-12,\'Original six-hour PFI replay mismatch\'\n                seal(check_path,{\'identity\':check_identity,\'status\':\'PASS\',\'probability_max_difference\':pdiff,\'importance_max_difference\':difference,\'baseline_difference\':bd})\n            w=original[\'warning_scores\'];rows.append({\'dataset\':dataset,\'seed\':seed,\'model\':model_name,\'hours\':6,\n                \'sampling_instability\':1-float(w[\'sampling_stability\']),\'environmental_instability\':float(w[\'environmental_instability\']),\n                \'sampling_environment_gap\':float(w[\'sampling_environment_gap\']),\'environmental_groups\':int(w[\'environmental_groups\']),\n                \'sampling_groups\':int(w[\'sampling_groups\']),\'score_origin\':\'historical_six_hour\'})\n            for hours,(actual,pseudo,membership,starts) in groups.items():\n                means={};undefined={};paircounts={}\n                for kind,codes in [(\'environmental\',actual),(\'sampling\',pseudo)]:\n                    vectors=[]\n                    for code,start in enumerate(starts):\n                        positions=np.flatnonzero(codes==code);row_hash=hashlib.sha256(env.row_id.to_numpy(np.int64)[positions].tobytes()).hexdigest()\n                        ji={**check_identity,\'group_type\':kind,\'selected_rows_sha256\':row_hash,\'rows\':len(positions),\'permutation_repeats\':3}\n                        # Width omitted from cache/RNG: identical row sets are computed once.\n                        job_path=out/\'groups\'/dataset/f\'seed_{seed}\'/model_name/(kind+\'_\'+row_hash+\'.json\')\n                        if job_path.exists():\n                            result=readsealed(job_path);assert result[\'identity\']==ji and result[\'status\']==\'PASS\';reused_jobs+=1\n                        else:\n                            print(f\'{completed_refs}/{total_refs} START {dataset} {seed} {model_name} {hours}h {kind} group {code+1}/{len(starts)} rows={len(positions)}\',flush=True)\n                            begin=time.perf_counter()\n                            pfi=permutation_importance(model,x[positions],y[positions],scoring=score_at(threshold),n_repeats=3,\n                                random_state=state(\'TEMPORAL_REGROUPING_FULL_V1\',dataset,seed,model_name,kind,row_hash,\'PFI\'),n_jobs=1)\n                            result={\'identity\':ji,\'status\':\'PASS\',\'importance_repeats\':pfi.importances.tolist(),\n                                \'baseline_balanced_accuracy\':float(score_at(threshold)(model,x[positions],y[positions])),\n                                \'elapsed_seconds\':time.perf_counter()-begin,\'created_utc\':datetime.now(timezone.utc).isoformat()}\n                            seal(job_path,result);new_jobs+=1\n                        importance=np.asarray(result[\'importance_repeats\']);assert importance.shape==(49,3) and np.isfinite(importance).all()\n                        vectors.append(importance.mean(axis=1))\n                        group_rows.append({\'dataset\':dataset,\'seed\':seed,\'model\':model_name,\'hours\':hours,\'group_type\':kind,\n                            \'group_start_ms\':int(start),\'rows\':len(positions),\'benign\':int(np.sum(y[positions]==0)),\n                            \'attack\':int(np.sum(y[positions]==1)),\'baseline_balanced_accuracy\':result[\'baseline_balanced_accuracy\'],\n                            \'checkpoint\':str(job_path.relative_to(out))})\n                        completed_refs+=1\n                        session.update(completed_group_references=completed_refs,new_group_jobs=new_jobs,reused_group_jobs=reused_jobs,current_dataset=dataset,current_seed=seed,current_model=model_name)\n                        write(out/\'progress.json\',session)\n                        print(f\'{completed_refs}/{total_refs} DONE; new={new_jobs}, cached={reused_jobs}\',flush=True)\n                    means[kind],undefined[kind],paircounts[kind]=paired_stability(vectors)\n                rows.append({\'dataset\':dataset,\'seed\':seed,\'model\':model_name,\'hours\':hours,\n                    \'sampling_instability\':1-means[\'sampling\'],\'environmental_instability\':1-means[\'environmental\'],\n                    \'sampling_environment_gap\':means[\'sampling\']-means[\'environmental\'],\n                    \'environmental_groups\':len(starts),\'sampling_groups\':len(starts),\n                    \'undefined_environmental_pairs\':undefined[\'environmental\'],\'undefined_sampling_pairs\':undefined[\'sampling\'],\n                    \'environmental_pairs\':paircounts[\'environmental\'],\'sampling_pairs\':paircounts[\'sampling\'],\n                    \'score_origin\':\'post_hoc_regrouped_original_rows\'})\n    assert completed_refs==total_refs\n    units=pd.DataFrame(rows);assert len(units)==180\n    # The audited CSE partition is identical at 12 and 24 hours: scores must match exactly.\n    a=units[(units.dataset==\'NF_CSE_CIC_IDS2018_v3\')&(units.hours==12)].sort_values([\'seed\',\'model\'])\n    b=units[(units.dataset==\'NF_CSE_CIC_IDS2018_v3\')&(units.hours==24)].sort_values([\'seed\',\'model\'])\n    assert np.array_equal(a[WARN].to_numpy(),b[WARN].to_numpy()),\'Identical CSE partitions should yield identical warnings\'\n    pd.DataFrame(group_rows).to_csv(out/\'group_inventory.csv\',index=False)\n    analysis=predictive_analysis(root,out,units)\n    session.update(status=\'FULL_PASS\' if analysis[\'rank_deficient_fits\']==0 else \'COMPLETED_WITH_RANK_DEFICIENCY\',\n        completed_utc=datetime.now(timezone.utc).isoformat(),analysis=analysis,identical_cse_12h_24h_verified=True,\n        original_frozen_files_modified=False,full_raw_data_alternative_width_experiment=False)\n    write(out/\'progress.json\',session)\n    write(out/\'output_checksums.json\',{str(p.relative_to(out)):sha(p) for p in out.rglob(\'*\') if p.is_file() and p.name!=\'output_checksums.json\' and p.suffix!=\'.tmp\'})\n    print(session[\'status\'],completed_refs,\'group references;\',analysis,flush=True)\n    return session\n'
exec(compile(FULL_SCRIPT,"paper3_temporal_grouping_full.py","exec"))


In [ ]:
import traceback
try:
    run_full(PAPER3_ROOT,OUTPUT,FULL_PLAN,FULL_SCRIPT)
except Exception as exc:
    write(OUTPUT/'last_error.json',{'status':'STOPPED','error':str(exc),'traceback':traceback.format_exc()})
    print('STOPPED. Completed group checkpoints remain saved. Inspect/upload the report before changing anything.')
    raise


In [ ]:
import zipfile
package = OUTPUT.with_suffix('.zip')
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob('*')):
        if path.is_file() and path.suffix != '.tmp':
            archive.write(path,arcname=str(path.relative_to(OUTPUT.parent)))
print('Saved ZIP:',package)
files.download(str(package))
